# 05 — Correspondence Generation (Feature Matching)

Runs SIFT (baseline), RoMa (dense robust matcher), and a MINIMA scaffold on
the scale-matched pairs from Notebook 04, and saves raw match-point CSVs.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, csv
import numpy as np
import cv2
import rasterio
import matplotlib.pyplot as plt

DRIVE_ROOT = '/content/drive/MyDrive/26166_DATA'
OUT_ROOT   = f'{DRIVE_ROOT}/processed/matches'
os.makedirs(OUT_ROOT, exist_ok=True)

with open(f'{DRIVE_ROOT}/processed/manifest_scalespace.csv') as f:
    pair_manifest = list(csv.DictReader(f))

for m in pair_manifest:
    print(m['pair'], '->', m['fine_scaled_path'], m['coarse_path'])


## 0. GPU check

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print('Runtime > Change runtime type > GPU, then re-run this cell.')


## 1. Baseline — classical SIFT matching

In [ ]:
def sift_match(img1, img2, ratio_thresh=0.75):
    sift = cv2.SIFT_create()
    kp1, des1 = sift.detectAndCompute(img1, None)
    kp2, des2 = sift.detectAndCompute(img2, None)
    if des1 is None or des2 is None:
        return np.empty((0, 2)), np.empty((0, 2)), []
    bf = cv2.BFMatcher()
    raw_matches = bf.knnMatch(des1, des2, k=2)
    good = [m for m, n in raw_matches if m.distance < ratio_thresh * n.distance]
    pts1 = np.array([kp1[m.queryIdx].pt for m in good])
    pts2 = np.array([kp2[m.trainIdx].pt for m in good])
    return pts1, pts2, good

def load_gray(path):
    with rasterio.open(path) as src:
        return src.read(1)

sift_results = {}
for m in pair_manifest:
    img_fine = load_gray(m['fine_scaled_path'])
    img_coarse = load_gray(m['coarse_path'])
    pts1, pts2, good = sift_match(img_fine, img_coarse)
    sift_results[m['pair']] = (pts1, pts2)
    print(f"{m['pair']}: SIFT found {len(good)} candidate matches")


## 2. RoMa — dense robust matcher

In [ ]:
!pip install -q git+https://github.com/Parskatt/RoMa.git

from romatch import roma_outdoor
device = 'cuda' if torch.cuda.is_available() else 'cpu'
roma_model = roma_outdoor(device=device)
print('RoMa loaded.')


In [ ]:
from PIL import Image

def roma_match(img1_path, img2_path, model, device, sample_n=5000):
    def ensure_rgb_png(path):
        arr = load_gray(path)
        rgb = np.stack([arr]*3, axis=-1)
        tmp_path = path.replace('.tif', '_rgb.png')
        Image.fromarray(rgb).save(tmp_path)
        return tmp_path

    p1, p2 = ensure_rgb_png(img1_path), ensure_rgb_png(img2_path)
    with rasterio.open(img1_path) as s: h1, w1 = s.height, s.width
    with rasterio.open(img2_path) as s: h2, w2 = s.height, s.width

    warp, certainty = model.match(p1, p2, device=device)
    matches, certainty_sampled = model.sample(warp, certainty, num=sample_n)
    kpts1, kpts2 = model.to_pixel_coordinates(matches, h1, w1, h2, w2)
    return kpts1.cpu().numpy(), kpts2.cpu().numpy(), certainty_sampled.cpu().numpy()

roma_results = {}
for m in pair_manifest:
    pts1, pts2, cert = roma_match(m['fine_scaled_path'], m['coarse_path'], roma_model, device)
    roma_results[m['pair']] = (pts1, pts2, cert)
    print(f"{m['pair']}: RoMa produced {len(pts1)} dense-sampled matches (mean certainty {cert.mean():.3f})")


## 3. MINIMA — modality-invariant matcher

Check `https://github.com/LSXI7/MINIMA`'s current README for exact demo-script
arguments before running (they change more often than the underlying library).


In [ ]:
!git clone -q https://github.com/LSXI7/MINIMA.git /content/MINIMA
%cd /content/MINIMA
!pip install -q -r requirements.txt
# Adapt per the repo's current Quick Start, e.g.:
# !python demo.py --im1 <fine_scaled_path> --im2 <coarse_path> --matcher loftr \
#     --ckpt weights/minima_loftr.ckpt --out /content/minima_out
%cd /content


## 4. Visualize matches for one pair

In [ ]:
def draw_matches(img1, img2, pts1, pts2, max_lines=100, title=''):
    h1, w1 = img1.shape; h2, w2 = img2.shape
    canvas = np.zeros((max(h1, h2), w1 + w2), dtype=np.uint8)
    canvas[:h1, :w1] = img1
    canvas[:h2, w1:w1 + w2] = img2
    canvas_rgb = cv2.cvtColor(canvas, cv2.COLOR_GRAY2BGR)
    idx = np.random.choice(len(pts1), size=min(max_lines, len(pts1)), replace=False) if len(pts1) else []
    for i in idx:
        p1 = tuple(np.round(pts1[i]).astype(int))
        p2 = tuple(np.round(pts2[i]).astype(int) + np.array([w1, 0]))
        color = tuple(np.random.randint(0, 255, 3).tolist())
        cv2.circle(canvas_rgb, p1, 3, color, -1)
        cv2.circle(canvas_rgb, p2, 3, color, -1)
        cv2.line(canvas_rgb, p1, p2, color, 1)
    plt.figure(figsize=(16, 8))
    plt.imshow(cv2.cvtColor(canvas_rgb, cv2.COLOR_BGR2RGB))
    plt.title(title); plt.axis('off'); plt.show()

first_pair = pair_manifest[0]['pair']
img1 = load_gray([m for m in pair_manifest if m['pair'] == first_pair][0]['fine_scaled_path'])
img2 = load_gray([m for m in pair_manifest if m['pair'] == first_pair][0]['coarse_path'])

sp1, sp2 = sift_results[first_pair]
draw_matches(img1, img2, sp1, sp2, title=f'{first_pair} — SIFT (raw, unfiltered)')

rp1, rp2, _ = roma_results[first_pair]
draw_matches(img1, img2, rp1, rp2, title=f'{first_pair} — RoMa (raw, unfiltered)')


## 5. Save raw match points for every pair + method

In [ ]:
def save_matches_csv(pts1, pts2, out_path, extra_cols=None):
    with open(out_path, 'w', newline='') as f:
        writer = csv.writer(f)
        header = ['x_fine', 'y_fine', 'x_coarse', 'y_coarse']
        if extra_cols: header += list(extra_cols.keys())
        writer.writerow(header)
        for i in range(len(pts1)):
            row = [pts1[i][0], pts1[i][1], pts2[i][0], pts2[i][1]]
            if extra_cols: row += [extra_cols[k][i] for k in extra_cols]
            writer.writerow(row)

for m in pair_manifest:
    pair = m['pair']
    out_dir = f'{OUT_ROOT}/{pair}'
    os.makedirs(out_dir, exist_ok=True)
    p1, p2 = sift_results[pair]
    save_matches_csv(p1, p2, f'{out_dir}/sift_raw_matches.csv')
    p1, p2, cert = roma_results[pair]
    save_matches_csv(p1, p2, f'{out_dir}/roma_raw_matches.csv', extra_cols={'certainty': cert})
    print(f"Saved raw match CSVs for {pair} (SIFT: {len(sift_results[pair][0])}, RoMa: {len(roma_results[pair][0])})")

print('\nRaw, unfiltered matches saved. Build 06_transform_and_warp.ipynb next')
print('for MAGSAC/RANSAC outlier rejection, sub-pixel refinement, warping,')
print('and the final registered product + evaluation metrics.')


---
### Notes
- Because OHRC↔TMC-2 now have *real* georeferencing from Notebook 03, you can
  cross-check RoMa/SIFT match quality against ground truth: convert each
  matched pixel pair to lon/lat using both images' transforms and see how far
  apart the two lon/lat estimates land — this gives you a real accuracy number
  even before building the full RANSAC/evaluation notebook.
- IIRS matches stay pixel-only until you inherit georeferencing transitively
  through TMC-2, as noted in Notebook 03.
